# RoomQuest 01｜驗證器與玩家模式

這份 Notebook 示範 RoomQuest 的兩個部分：

1. **Python 驗證器**：檢查關卡是否引用存在的物品、條件是否唯一、答案是否來自固定的場景資料。
2. **玩家模式**：用 Gradio 試玩通過驗證的遊戲。遊玩過程**不呼叫 AI**，答案只留在伺服器端。

本份不需要任何 API 金鑰。請由上往下，一格一格執行。

## 1. 下載程式與安裝套件

`REPO_URL` 請改成你的 GitHub repository。套件版本和課程一致（`gradio==6.30.0`）。

In [ ]:
REPO_URL = "https://github.com/lyixuan0718/roomquest.git"

!git clone -q {REPO_URL} roomquest
%cd roomquest
%pip install -q -r requirements.txt

## 2. 執行測試

每一種錯誤都有一個「故意改壞一處」的測試。全部通過，代表驗證器真的抓得到這些問題。

In [ ]:
!python -m pytest -q

## 3. 驗證示範場景

B305 是**示意案例**（不是真實教室）。看驗證報告：✗ 是錯誤（會擋下關卡），△ 是警告（交給主持人判斷）。

In [ ]:
from roomquest.verifier import load_json, verify

SCENE_DIR = "data/scenes/classroom-b305"
scene = load_json(f"{SCENE_DIR}/scene.json")
game = load_json(f"{SCENE_DIR}/game.json")

report = verify(scene, game)
print(report.to_text())

## 4. 故意改壞一關，看驗證器的反應

把第 2 關的條件改成「窗邊、紅色」：水壺和書包都符合，所以會出現 `AMBIGUOUS_TARGET`。

`to_feedback()` 是之後要交給 AI 修改關卡的文字：只列出失敗的關卡與原因。

In [ ]:
import copy

broken = copy.deepcopy(game)
broken["levels"][1]["references"][1]["constraints"] = {"region": "窗邊", "color": "紅色"}

bad_report = verify(scene, broken)
print(bad_report.to_text())
print()
print(bad_report.to_feedback())

## 5. 啟動玩家模式

執行後點公開連結，用手機開啟就能試玩。連結約一週後失效。遊玩紀錄會寫到 `logs/plays/`。

In [ ]:
from roomquest.player_ui import build_player_app

demo = build_player_app(f"{SCENE_DIR}/scene.json", f"{SCENE_DIR}/game.json", log_dir="logs/plays")
demo.launch(share=True, debug=True)